# ToDo

1. Среднее значение без и с сопротивлением:
- (левая вдох + левая выдох + правая вдох + правая выдох) / 4

# Подготовка

## Настройка графики

In [ ]:
# windows.options(height=5.4, width=7)
oldpar <- par()
par(mar = c(8, 4, 1, 2), "xpd" = FALSE)
options(repr.plot.height = 9, repr.plot.width = 12)
options(warn = -1)

## Библиотеки

In [ ]:
options(java.parameters = "-Xmx4096m")

require(readxl, quietly = TRUE, warn.conflicts = FALSE)

require(vcd, quietly = TRUE, warn.conflicts = FALSE)
require(coin, quietly = TRUE, warn.conflicts = FALSE)
# independence_test
require(agricolae, quietly = TRUE, warn.conflicts = FALSE)
# HSD.test
require(pgirmess, quietly = TRUE, warn.conflicts = FALSE)
# kruskalmc
require(nortest, quietly = TRUE, warn.conflicts = FALSE)
# for normality test in case of N>5000. ad.test -- Anderson-Darling normality test
require(RcmdrMisc, quietly = TRUE, warn.conflicts = FALSE)
# numSumm

require(beeswarm, quietly = TRUE, warn.conflicts = FALSE)
require(lattice, quietly = TRUE, warn.conflicts = FALSE)
require(mosaic, quietly = TRUE, warn.conflicts = FALSE)
require(ggplot2, quietly = TRUE, warn.conflicts = FALSE)
require(ggpubr, quietly = TRUE, warn.conflicts = FALSE)
# ggqqplot
# require(ggExtra, quietly = TRUE, warn.conflicts = FALSE);
# require(gridExtra, quietly = TRUE, warn.conflicts = FALSE);
# require(ggfortify, quietly = TRUE, warn.conflicts = FALSE);
require(ggalluvial, quietly = TRUE)
# flow diagramm
require(hrbrthemes, quietly = TRUE, warn.conflicts = FALSE)
# ggparcoord
require(GGally, quietly = TRUE, warn.conflicts = FALSE)
# ggparcoord
require(viridis, quietly = TRUE, warn.conflicts = FALSE)
# ggparcoord


require(rstatix, quietly = TRUE)
# identify_outliers
require(dplyr, quietly = TRUE, warn.conflicts = FALSE)
require(tidyr, quietly = TRUE, warn.conflicts = FALSE)
require(tidycmprsk, quietly = TRUE, warn.conflicts = FALSE)
# require(tidyverse, quietly = TRUE, warn.conflicts = FALSE);

require(IRdisplay, quietly = TRUE, warn.conflicts = FALSE)
require(repr, quietly = TRUE, warn.conflicts = FALSE)
# require(TeachingDemos, quietly = TRUE, warn.conflicts = FALSE);
# require(Rmisc, quietly = TRUE, warn.conflicts = FALSE);
# require(ranger, quietly = TRUE, warn.conflicts = FALSE);
# require(Epi, quietly = TRUE, warn.conflicts = FALSE);
# require(car, quietly = TRUE, warn.conflicts = FALSE);
# require(exactRankTests, quietly = TRUE, warn.conflicts = FALSE);
# require(abind, quietly = TRUE, warn.conflicts = FALSE);
# require(mstate, quietly = TRUE, warn.conflicts = FALSE);
# require(gnm, quietly = TRUE, warn.conflicts = FALSE);
# require(multcomp, quietly = TRUE, warn.conflicts = FALSE);
# require(scales, quietly = TRUE, warn.conflicts = FALSE);
# require(Rcmdr, quietly = TRUE, warn.conflicts = FALSE);
# require(tigerstats, quietly = TRUE, warn.conflicts = FALSE);
# require(fpp3, quietly = TRUE, warn.conflicts = FALSE);
# require(tsibble, quietly = TRUE, warn.conflicts = FALSE);
# require(lubridate, quietly = TRUE, warn.conflicts = FALSE);
# require(cowplot, quietly = TRUE, warn.conflicts = FALSE);
# require(smplot2, quietly = TRUE, warn.conflicts = FALSE);
# require(biostat3, quietly = TRUE, warn.conflicts = FALSE);
# require(data.table, quietly = TRUE, warn.conflicts = FALSE);

## Функции

In [ ]:
build_sankey <- function(data, group, parameter, group_colors) {
  for (groupID in parameter) {
    groupSize <- nlevels(data[[groupID]])
    group_colors1 <- group_colors[1:groupSize]
    for (i in 2:nlevels(data[[group]])) {
      tmp_tab <- table(data[data[[group]] == levels(data[[group]])[i - 1], groupID], data[data[[group]] == levels(data[[group]])[i], groupID])
      colnames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i], 2, "left", "0"), colnames(tmp_tab), sep = "_")
      rownames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i - 1], 2, "left", "0"), rownames(tmp_tab), sep = "_")
      if (i == 2) {
        out_tab <- as.data.table(tmp_tab)
        colnames(out_tab) <- c("source", "target", "value")
      } else {
        out_tab <- rbind(out_tab, as.data.table(tmp_tab), use.names = FALSE)
      }
    }
    out_tab <- cbind(as.data.frame(out_tab), group = rep(group_colors1, ceiling(dim(out_tab)[1] / length(group_colors1)))[1:dim(out_tab)[1]])

    links <- data.frame(out_tab[out_tab$value != 0, ])
    links <- links[order(links$source), ]

    nodes <- data.frame(name = unique(c(as.character(links$source), as.character(links$target))))
    unique_status <- sort(unique(substring(nodes$name, unlist(gregexpr(".[0-9]+$", gsub("\\+", ".", nodes$name))) + 1)))
    nodes$group <- group_colors1[match(gsub(".*\\.([0-9]+)$", "\\1", gsub("\\+", ".", nodes$name)), unique_status)]
    nodes <- nodes[order(nodes$name), ]
    rowTotals <- unique(rbind(aggregate(links$value, by = list(links$source), FUN = sum), aggregate(links$value, by = list(links$target), FUN = sum)))
    names(rowTotals) <- c("group", "x")
    nodes$rowtotal <- rowTotals$x
    nodes <- transform(nodes, rowpc = ave(rowtotal, substring(nodes$name, 1, 3), FUN = prop.table) * 100)

    links$IDsource <- match(links$source, nodes$name) - 1
    links$IDtarget <- match(links$target, nodes$name) - 1

    color <- sprintf(
      "d3.scaleOrdinal().domain([%s]).range([%s]);",
      paste0("'", paste(nodes$name, collapse = "", ""), "'"),
      paste0("'", paste(nodes$group, collapse = "", ""), "'")
    )

    #    links$group = substring(links$group, 2)
    #    nodes$group = substring(nodes$group, 2)

    sn <- sankeyNetwork(
      Links = links, Nodes = nodes,
      Source = "IDsource", Target = "IDtarget", Value = "value",
      NodeID = "name", LinkGroup = "source", NodeGroup = "name",
      units = "cases",
      sinksRight = FALSE, nodeWidth = 30, nodePadding = 30,
      colourScale = color,
      fontSize = 14, fontFamily = "Arial"
    )
    print(sn)
  }
}

metrics <- c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "s110", "m_s110")
# group_colors = c("#f6e1f7ff", "#ecaad6FF", "#de6bb8ff", "#cc2e97ff", "#9d40a3FF", "#a7157FF", "#82003EFF", "#5e002dFF")
group_colors <- c("#0084ffff", "#44bec7ff", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157fff", "#82003eff", "#5e002dff", "#000000ff")
metrics <- c("d330")
# build_sankey(preml, "time", metrics, group_colors)


build_sankeyP <- function(data, group, parameter, group_colors) {
  for (groupID in parameter) {
    groupSize <- nlevels(data[[groupID]])
    group_colors1 <- group_colors[1:groupSize]
    for (i in 2:nlevels(data[[group]])) {
      tmp_tab <- table(data[data[[group]] == levels(data[[group]])[i - 1], groupID], data[data[[group]] == levels(data[[group]])[i], groupID])
      colnames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i], 2, "left", "0"), colnames(tmp_tab), sep = "_")
      rownames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i - 1], 2, "left", "0"), rownames(tmp_tab), sep = "_")
      if (i == 2) {
        out_tab <- as.data.table(tmp_tab)
        colnames(out_tab) <- c("source", "target", "value")
      } else {
        out_tab <- rbind(out_tab, as.data.table(tmp_tab), use.names = FALSE)
      }
    }
    out_tab <- cbind(as.data.frame(out_tab), group = rep(group_colors1, ceiling(dim(out_tab)[1] / length(group_colors1)))[1:dim(out_tab)[1]])

    links <- data.frame(out_tab[out_tab$value != 0, ])
    links <- links[order(links$source), ]

    nodes <- data.frame(name = unique(c(as.character(links$source), as.character(links$target))))
    unique_status <- sort(unique(substring(nodes$name, unlist(gregexpr(".[0-9]+$", gsub("\\+", ".", nodes$name))) + 1)))
    nodes$group <- group_colors1[match(gsub(".*\\.([0-9]+)$", "\\1", gsub("\\+", ".", nodes$name)), unique_status)]
    nodes <- nodes[order(nodes$name), ]
    rowTotals <- unique(rbind(aggregate(links$value, by = list(links$source), FUN = sum), aggregate(links$value, by = list(links$target), FUN = sum)))
    names(rowTotals) <- c("group", "x")
    nodes$rowtotal <- rowTotals$x
    nodes <- transform(nodes, rowpc = ave(rowtotal, substring(nodes$name, 1, 3), FUN = prop.table) * 100)

    links$IDsource <- match(links$source, nodes$name) - 1
    links$IDtarget <- match(links$target, nodes$name) - 1

    link <- list(
      source = links$IDsource,
      target = links$IDtarget,
      value = links$value,
      color = links$group
    )
    node <- list(
      label = sprintf("%i (%2.1f%%)", nodes$rowtotal, nodes$rowpc),
      color = nodes$group,
      pad = 10,
      line = list(
        color = "black",
        width = 1
      ),
      thickness = 30
    )
    domain <- list(
      x = c(0, 1),
      y = c(0, 1)
    )

    p <- plot_ly(
      link = link,
      node = node,
      domain = domain,
      type = "sankey",
      orientation = "h",
      valueformat = " .0f "
      # , valuesuffix = " случаи"
      , alpha = 1,
      height = 650,
      width = 850
    )

    p <- config(
      p,
      scrollZoom = TRUE,
      responsive = TRUE
      # , staticPlot = TRUE
      # , displayModeBar = TRUE
      , displaylogo = FALSE
    ) %>% layout(
      autosize = TRUE,
      title = groupID,
      font = list(size = 14),
      margin = list(
        l = 50,
        r = 50,
        b = 0,
        t = 100
        # , pad = 4
        # , automargin = TRUE
      ),
      xaxis = list(title = "Sepal Length (cm)"),
      legend = list(x = 1, y = 0.5)
    )
    print(p)
  }
}

# group_colors = c("#0084ff99", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("#0084ff99", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("rgba(0,255,255,0.4)", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("#0084ffff", "#44bec7FF", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157ff", "#82003eff", "#5e002dff")
metrics <- c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "s110", "m_s110")
group_colors <- c("#0084ffff", "#44bec7ff", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157fff", "#82003eff", "#5e002dff", "#000000ff")
metrics <- c("b750")


# build_sankeyP(lorl, "time", metrics, group_colors)


build_sankey_strataG <- function(data, group, strata, parameter, group_colors, save_multiple, save_file) {
  if (save_multiple) {
    pdf(sprintf("%s.pdf", strata), onefile = TRUE, paper = "a4r", width = 20, height = 14, encoding = "KOI8-R")
  }
  for (groupID in parameter) {
    in_data <- data[!is.na(data[[groupID]]), ]
    p <- ggplot(
      in_data,
      aes(
        x = in_data[[group]],
        stratum = in_data[[groupID]],
        alluvium = in_data[["uid"]],
        fill = in_data[[groupID]]
      )
      # , width = 1980
      # , height = 1024
    ) +
      facet_wrap(
        in_data[[strata]],
        scales = "free",
        strip.position = "top",
        drop = TRUE
      ) +
      geom_flow(
        # stat = "alluvium"
        # , lode.guidance = "frontback"
        ,
        width = 0.27,
        color = "darkgray",
        curve_type = "arctangent" # "linear", "cubic", "quintic", "sine", "arctangent", and "sigmoid". "xspline"
      ) +
      geom_stratum(alpha = .5) +
      geom_text(
        stat = "flow",
        aes(
          label = sprintf(" %i ", after_stat(n)),
          hjust = (after_stat(flow) == "to"),
          vjust = (after_stat(flow) == "to")
        ),
        size = 4
      ) +
      geom_text(
        stat = "stratum",
        aes(
          label = sprintf("%i\n(%3.1f%%)", after_stat(n), after_stat(prop * 100))
          # , hjust = as.integer(after_stat(stratum)) - 2
        ),
        hjust = 0,
        size = 5,
        nudge_x = 0.2
      ) +
      # expand_limits(x = 0, y = 0) +
      xlab(
        group
      ) +
      labs(fill = groupID, size = 5) +
      theme(
        legend.position = "bottom",
        panel.background = element_rect(fill = NA),
        panel.border = element_blank(),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        strip.text = element_text(size = 14), ,
        axis.ticks.y = element_blank(),
        axis.text.y = element_blank(),
        axis.text.x = element_text(size = 12),
        axis.title.x = element_text(size = 14),
        axis.line.x = element_line(size = 1, colour = "black", linetype = 1),
        legend.text = element_text(size = 14),
        legend.title = element_blank(),
        panel.spacing.x = unit(0.5, "lines"),
        text = element_text(family = "Arial Narrow")
      ) +
      scale_fill_manual(values = group_colors) +
      ggtitle(groupID)
    if (save_file) {
      ggsave(sprintf("%s_%s.png", strata, groupID), plot = p, width = 36, height = 24, unit = "cm", dpi = "print")
    }
    print(p)
  }
  if (save_multiple) {
    dev.off()
  }
}


build_sankey_strata <- function(data, group, strata, parameter, group_colors) {
  strata_data <- list()
  for (i in 1:nlevels(data[[strata]])) {
    tmp_data <- subset(data, data[[strata]] == levels(data[[strata]])[i])
    strata_data[[i]] <- tmp_data
  }
  for (groupID in parameter) {
    for (i in seq_along(strata_data)) {
      print(paste(levels(data[[strata]])[i], ", ", groupID))
      build_sankey(strata_data[[i]], group, groupID, group_colors)
    }
  }
}

metrics <- c("m_s110")
# metrics = c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "m_s110")
group_colors <- c("#0084ff", "#44bec7", "#ffc300", "#fa3c4c", "#d696bb", "#a7157f", "#82003e", "#5e002d", "#000000")

# build_sankey_strataG(preml, "time", "Абилитация", metrics, group_colors, FALSE, TRUE)

## Данные

### Загрузка

In [ ]:
# sessionInfo()
# options(encoding = "UTF-8")
lor <- read_excel("C:\\Analysis\\OTOLARING\\Nidelko\\lor.xlsx", sheet = "данные")
lor <- as.data.frame(lor)

### Преобразование

#### Отбор данных

In [ ]:
# SNOT 22
lor <- lor %>%
  select(c(
    "uid", "группа", "id", "рождение", "возраст", "пол",
    "длительность заболевания", "удаление полипов в анамнезе", "хронический ринит", "АР", "БА",
    "искривление носовой перегородки", "повторная аллотрансплантация", "оперированные пазухи", "верхнечелюстная", "этмоидальная", "фронтальная", "сфеноидальная", "FESS",
    "вазотомия нижних носовых раковин", "операция на перегородке носа", "распространенность полипозного процесса КТ",
    "лобные пазухи", "клиновидные пазухи", "решетчатый лабиринт передний", "решетчатый лабиринт задний", "верхнечелюстные пазухи", "остиомеатальный комплекс", "шкала Лунд-Маккей",
    "отделяемое", "корки", "отек", "бальная оценка",
    "левая половина вдох, скорость.0", "левая половина вдох, скорость.2", "левая половина вдох, скорость.3",
    "левая половина выдох, скорость.0", "левая половина выдох, скорость.2", "левая половина выдох, скорость.3",
    "правая половина вдох, скорость.0", "правая половина вдох, скорость.2", "правая половина вдох, скорость.3",
    "правая половина выдох, скорость.0", "правая половина выдох, скорость.2", "правая половина выдох, скорость.3",
    "сопротивление 150 ПА правая вдох.0", "сопротивление 150 ПА правая вдох.2", "сопротивление 150 ПА правая вдох.3",
    "сопротивление 150 ПА правая выдох.0", "сопротивление 150 ПА правая выдох.2", "сопротивление 150 ПА правая выдох.3",
    "сопротивление 150 ПА левая вдох.0", "сопротивление 150 ПА левая вдох.2", "сопротивление 150 ПА левая вдох.3",
    "сопротивление 150 ПА левая выдох.0", "сопротивление 150 ПА левая выдох.2", "сопротивление 150 ПА левая выдох.3"
  ))
lor <- as.data.frame(lor)

#### Контрасты

In [ ]:
lor$группа <- factor(lor$группа, c("ОГ1", "ОГ2", "КГ"))
lor$пол <- factor(lor$пол)

lor$"хронический ринит" <- factor(lor$"хронический ринит")
lor$"АР" <- factor(lor$"АР")
lor$"БА" <- factor(lor$"БА")

lor$"искривление носовой перегородки" <- factor(lor$"искривление носовой перегородки")
lor$"повторная аллотрансплантация" <- factor(lor$"повторная аллотрансплантация")
lor$"верхнечелюстная" <- factor(lor$"верхнечелюстная", c("нет", "односторонняя", "двустороняя"))
lor$"этмоидальная" <- factor(lor$"этмоидальная", c("нет", "односторонняя", "двустороняя"))
lor$"фронтальная" <- factor(lor$"фронтальная", c("нет", "односторонняя", "двустороняя"))
lor$"сфеноидальная" <- factor(lor$"сфеноидальная", c("нет", "односторонняя", "двустороняя"))
lor$"вазотомия нижних носовых раковин" <- factor(lor$"вазотомия нижних носовых раковин")
lor$"операция на перегородке носа" <- factor(lor$"операция на перегородке носа")
lor$FESS <- factor(lor$FESS)

lor$"распространенность полипозного процесса КТ" <- factor(lor$"распространенность полипозного процесса КТ")
lor$"лобные пазухи" <- factor(lor$"лобные пазухи")
lor$"клиновидные пазухи" <- factor(lor$"клиновидные пазухи")
lor$"решетчатый лабиринт передний" <- factor(lor$"решетчатый лабиринт передний")
lor$"решетчатый лабиринт задний" <- factor(lor$"решетчатый лабиринт задний")
lor$"верхнечелюстные пазухи" <- factor(lor$"верхнечелюстные пазухи")
lor$"остиомеатальный комплекс" <- factor(lor$"остиомеатальный комплекс")

lor$"отделяемое" <- factor(lor$"отделяемое")
lor$"корки" <- factor(lor$"корки")
lor$"отек" <- factor(lor$"отек")

#### Суммарные

In [ ]:
lor <- lor %>%
  mutate(
    `скорость слева.0` = (`левая половина вдох, скорость.0` + `левая половина выдох, скорость.0`) / 2,
    `скорость слева.2` = (`левая половина вдох, скорость.2` + `левая половина выдох, скорость.2`) / 2,
    `скорость слева.3` = (`левая половина вдох, скорость.3` + `левая половина выдох, скорость.3`) / 2,
    `скорость справа.0` = (`правая половина вдох, скорость.0` + `правая половина выдох, скорость.0`) / 2,
    `скорость справа.2` = (`правая половина вдох, скорость.2` + `правая половина выдох, скорость.2`) / 2,
    `скорость справа.3` = (`правая половина вдох, скорость.3` + `правая половина выдох, скорость.3`) / 2,
    `скорость.0` = (`скорость слева.0` + `скорость справа.0`) / 2,
    `скорость.2` = (`скорость слева.2` + `скорость справа.2`) / 2,
    `скорость.3` = (`скорость слева.3` + `скорость справа.3`) / 2,
    `сопротивление 150 ПА слева.0` = (`сопротивление 150 ПА левая вдох.0` + `сопротивление 150 ПА левая выдох.0`) / 2,
    `сопротивление 150 ПА слева.2` = (`сопротивление 150 ПА левая вдох.2` + `сопротивление 150 ПА левая выдох.2`) / 2,
    `сопротивление 150 ПА слева.3` = (`сопротивление 150 ПА левая вдох.3` + `сопротивление 150 ПА левая выдох.3`) / 2,
    `сопротивление 150 ПА справа.0` = (`сопротивление 150 ПА правая вдох.0` + `сопротивление 150 ПА правая выдох.0`) / 2,
    `сопротивление 150 ПА справа.2` = (`сопротивление 150 ПА правая вдох.2` + `сопротивление 150 ПА правая выдох.2`) / 2,
    `сопротивление 150 ПА справа.3` = (`сопротивление 150 ПА правая вдох.3` + `сопротивление 150 ПА правая выдох.3`) / 2,
    `сопротивление 150 ПА.0` = (`сопротивление 150 ПА слева.0` + `сопротивление 150 ПА справа.0`) / 2,
    `сопротивление 150 ПА.2` = (`сопротивление 150 ПА слева.2` + `сопротивление 150 ПА справа.2`) / 2,
    `сопротивление 150 ПА.3` = (`сопротивление 150 ПА слева.3` + `сопротивление 150 ПА справа.3`) / 2
  )

#### Длинная таблица

In [ ]:
lor <- as.data.frame(lor)
lorl <- reshape(lor,
  direction = "long",
  idvar = "uid",
  sep = ".",
  varying = list(
    34:36, 37:39, 40:42, 43:45,
    46:48, 49:51, 52:54, 55:57,
    58:60, 61:63, 64:66, 67:69, 70:72, 73:75
  ),
  v.names = c(
    "левая половина вдох, скорость", "левая половина выдох, скорость", "правая половина вдох, скорость", "правая половина выдох, скорость",
    "сопротивление 150 ПА правая вдох", "сопротивление 150 ПА правая выдох", "сопротивление 150 ПА левая вдох", "сопротивление 150 ПА левая выдох",
    "скорость слева", "скорость справа", "скорость", "сопротивление 150 ПА слева", "сопротивление 150 ПА справа", "сопротивление 150 ПА"
  ),
  timevar = "время",
  times = c("до операции", "3 мес.", "1 год")
)
lorl$"время" <- factor(lorl$"время", c("до операции", "3 мес.", "1 год"))

### Подключение

In [ ]:
try(detach(lor), silent = TRUE)
attach(lor)

# Общий анализ

In [ ]:
groupping_variable <- "группа"

## левая половина вдох, скорость.0

### Общее

In [ ]:
parname <- "левая половина вдох, скорость.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## левая половина вдох, скорость.2

### Общее

In [ ]:
parname <- "левая половина вдох, скорость.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## левая половина вдох, скорость.3

### Общее

In [ ]:
parname <- "левая половина вдох, скорость.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## левая половина выдох, скорость.0

### Общее

In [ ]:
parname <- "левая половина выдох, скорость.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## левая половина выдох, скорость.2

### Общее

In [ ]:
parname <- "левая половина выдох, скорость.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## левая половина выдох, скорость.3

### Общее

In [ ]:
parname <- "левая половина выдох, скорость.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## правая половина вдох, скорость.0

### Общее

In [ ]:
parname <- "правая половина вдох, скорость.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## правая половина вдох, скорость.2

### Общее

In [ ]:
parname <- "правая половина вдох, скорость.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## правая половина вдох, скорость.3

### Общее

In [ ]:
parname <- "правая половина вдох, скорость.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## правая половина выдох, скорость.0

### Общее

In [ ]:
parname <- "правая половина выдох, скорость.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## правая половина выдох, скорость.2

### Общее

In [ ]:
parname <- "правая половина выдох, скорость.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## правая половина выдох, скорость.3

### Общее

In [ ]:
parname <- "правая половина выдох, скорость.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА правая вдох.0

### Общее

In [ ]:
parname <- "сопротивление 150 ПА правая вдох.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА правая вдох.2

### Общее

In [ ]:
parname <- "сопротивление 150 ПА правая вдох.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА правая вдох.3

### Общее

In [ ]:
parname <- "сопротивление 150 ПА правая вдох.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА правая выдох.0

### Общее

In [ ]:
parname <- "сопротивление 150 ПА правая выдох.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА правая выдох.2

### Общее

In [ ]:
parname <- "сопротивление 150 ПА правая выдох.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА правая выдох.3

### Общее

In [ ]:
parname <- "сопротивление 150 ПА правая выдох.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА левая вдох.0

### Общее

In [ ]:
parname <- "сопротивление 150 ПА левая вдох.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА левая вдох.2

### Общее

In [ ]:
parname <- "сопротивление 150 ПА левая вдох.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА левая вдох.3

### Общее

In [ ]:
parname <- "сопротивление 150 ПА левая вдох.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА левая выдох.0

### Общее

In [ ]:
parname <- "сопротивление 150 ПА левая выдох.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА левая выдох.2

### Общее

In [ ]:
parname <- "сопротивление 150 ПА левая выдох.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА левая выдох.3

### Общее

In [ ]:
parname <- "сопротивление 150 ПА левая выдох.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость слева.0

### Общее

In [ ]:
parname <- "скорость слева.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
parname
print(names(lorl))
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость слева.2

### Общее

In [ ]:
parname <- "скорость слева.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость слева.3

### Общее

In [ ]:
parname <- "скорость слева.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость справа.0

### Общее

In [ ]:
parname <- "скорость справа.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость справа.2

### Общее

In [ ]:
parname <- "скорость справа.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость справа.3

### Общее

In [ ]:
parname <- "скорость справа.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость.0

### Общее

In [ ]:
parname <- "скорость.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость.2

### Общее

In [ ]:
parname <- "скорость.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## скорость.3

### Общее

In [ ]:
parname <- "скорость.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА слева.0

### Общее

In [ ]:
parname <- "сопротивление 150 ПА слева.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА слева.2

### Общее

In [ ]:
parname <- "сопротивление 150 ПА слева.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА слева.3

### Общее

In [ ]:
parname <- "сопротивление 150 ПА слева.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА справа.0

### Общее

In [ ]:
parname <- "сопротивление 150 ПА справа.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА справа.2

### Общее

In [ ]:
parname <- "сопротивление 150 ПА справа.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА справа.3

### Общее

In [ ]:
parname <- "сопротивление 150 ПА справа.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА.0

### Общее

In [ ]:
parname <- "сопротивление 150 ПА.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА.2

### Общее

In [ ]:
parname <- "сопротивление 150 ПА.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## сопротивление 150 ПА.3

### Общее

In [ ]:
parname <- "сопротивление 150 ПА.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = parname)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### График

In [ ]:
parname <- sub("\\..*", "", parname)
max(lorl[[parname]])
min(lorl[[parname]])
beeswarm(lorl[[parname]] ~ lorl$группа + lorl$"время",
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$группа + lorl$"время", add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0.5, 0.5, 0.3, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(6, min(lorl[[parname]]) / 2.3, levels(lorl$пол), col = 1:nlevels(lorl$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)